# Digital Image Processing Lab - Assignment 10
**Problem Statements:**
1. **Write a program to denoise the input image using adaptive median filter.**
   - Input the initial window size and maximum size of window ($S_{\text{max}}$) from the user.
   - Compare the performance of median filter and adaptive median filter, using the PSNR values.
2. **Write a program to threshold the image using Otsu's global threshold for segmentation of input gray scale image.**

---
### Theoretical Principles
1. **Adaptive Median Filter (AMF):**
   Standard median filtering fails when impulse noise density is high ($> 20\%$) because the median itself can become an impulse. The Adaptive Median Filter dynamically expands window size up to $S_{\text{max}}$:
   - **Level A:**
     - $A_1 = Z_{\text{med}} - Z_{\text{min}}$
     - $A_2 = Z_{\text{med}} - Z_{\text{max}}$
     - If $A_1 > 0$ and $A_2 < 0$, go to **Level B** ($Z_{\text{med}}$ is not an impulse).
     - Else, increase window size. If window size $\le S_{\text{max}}$, repeat Level A; else output $Z_{\text{med}}$.
   - **Level B:**
     - $B_1 = Z_{xy} - Z_{\text{min}}$
     - $B_2 = Z_{xy} - Z_{\text{max}}$
     - If $B_1 > 0$ and $B_2 < 0$, output $Z_{xy}$ ($Z_{xy}$ is uncorrupted).
     - Else output $Z_{\text{med}}$.

2. **Otsu's Global Thresholding:**
   Maximizes the between-class variance $\sigma_B^2(k)$:
   $$\sigma_B^2(k) = \frac{\left[m_G P_1(k) - m(k)\right]^2}{P_1(k)(1 - P_1(k))}$$
   Optimal threshold $k^* = \arg\max_k \sigma_B^2(k)$.


In [ ]:
# Cell 1: Import Libraries
import numpy as np
import matplotlib.pyplot as plt
import cv2

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['image.cmap'] = 'gray'


In [ ]:
# Cell 2: Test Image & Heavy Salt-and-Pepper Noise
def get_clean_cameraman_like():
    x = np.linspace(-3, 3, 256)
    y = np.linspace(-3, 3, 256)
    xx, yy = np.meshgrid(x, y)
    base = (np.sin(xx) * np.cos(yy) * 80 + 128).astype(np.uint8)
    cv2.circle(base, (70, 70), 45, 240, -1)
    cv2.rectangle(base, (130, 130), (220, 220), 40, -1)
    cv2.putText(base, "OTSU", (50, 180), cv2.FONT_HERSHEY_SIMPLEX, 1.2, 200, 3)
    return base

clean_img = get_clean_cameraman_like()

def add_salt_and_pepper(image, salt_prob=0.15, pepper_prob=0.15):
    """Simulates high-density salt-and-pepper impulse noise (total 30%)."""
    noisy = image.copy()
    num_salt = int(salt_prob * image.size)
    num_pepper = int(pepper_prob * image.size)
    
    # Salt (white pixels = 255)
    coords_salt = [np.random.randint(0, i, num_salt) for i in image.shape]
    noisy[tuple(coords_salt)] = 255
    
    # Pepper (black pixels = 0)
    coords_pepper = [np.random.randint(0, i, num_pepper) for i in image.shape]
    noisy[tuple(coords_pepper)] = 0
    return noisy

np.random.seed(42)
noisy_sp = add_salt_and_pepper(clean_img, 0.15, 0.15)
print("30% Salt & Pepper noisy image generated.")


In [ ]:
# Cell 3: Task 1 - Adaptive Median Filter Implementation
initial_window_size = 3 #@param [3, 5] {type:"raw"}
max_window_size = 7 #@param [5, 7, 9, 11] {type:"raw"}

def adaptive_median_filter(image, s_init=3, s_max=7):
    """Implementation of Gonzalez & Woods Adaptive Median Filter."""
    h, w = image.shape
    output = np.zeros((h, w), dtype=np.uint8)
    max_pad = s_max // 2
    padded = np.pad(image, max_pad, mode='edge')
    
    for i in range(h):
        for j in range(w):
            z_xy = int(image[i, j])
            k = s_init
            while k <= s_max:
                rad = k // 2
                # Extract neighborhood
                pi = i + max_pad
                pj = j + max_pad
                win = padded[pi - rad : pi + rad + 1, pj - rad : pj + rad + 1]
                
                z_min = int(np.min(win))
                z_max = int(np.max(win))
                z_med = int(np.median(win))
                
                # Level A
                a1 = z_med - z_min
                a2 = z_med - z_max
                if a1 > 0 and a2 < 0:
                    # Level B
                    b1 = z_xy - z_min
                    b2 = z_xy - z_max
                    if b1 > 0 and b2 < 0:
                        output[i, j] = z_xy
                    else:
                        output[i, j] = z_med
                    break
                else:
                    k += 2
            if k > s_max:
                output[i, j] = z_med
                
    return output

# Apply Adaptive Median
denoised_amf = adaptive_median_filter(noisy_sp, initial_window_size, max_window_size)

# Apply Standard Median Filter for comparison
denoised_std_med = cv2.medianBlur(noisy_sp, initial_window_size)

def get_psnr(orig, test):
    mse = np.mean((orig.astype(float) - test.astype(float)) ** 2)
    return 10 * np.log10(255.0**2 / mse) if mse > 0 else float('inf')

psnr_noisy = get_psnr(clean_img, noisy_sp)
psnr_std = get_psnr(clean_img, denoised_std_med)
psnr_amf = get_psnr(clean_img, denoised_amf)

print(f"PSNR (Noisy 30% Impulse):          {psnr_noisy:.2f} dB")
print(f"PSNR (Standard Median {initial_window_size}x{initial_window_size}):       {psnr_std:.2f} dB")
print(f"PSNR (Adaptive Median [init={initial_window_size}, max={max_window_size}]): {psnr_amf:.2f} dB")
print(f"Improvement of AMF over Standard Median: {psnr_amf - psnr_std:.2f} dB")


In [ ]:
# Cell 4: Task 2 - Otsu's Global Thresholding From Scratch
def otsu_threshold_scratch(image):
    """Otsu global binarization algorithm implemented from scratch."""
    h, w = image.shape
    total_pixels = h * w
    
    # 1. Histogram & Normalized PDF
    hist = np.bincount(image.ravel(), minlength=256)
    p = hist / float(total_pixels)
    
    # 2. Cumulative Sums and Cumulative Means
    omega = np.cumsum(p)
    mu = np.cumsum(p * np.arange(256))
    mu_global = mu[-1]
    
    # 3. Between-Class Variance sigma_b^2 for all k in [0, 255]
    sigma_b_squared = np.zeros(256)
    
    for k in range(256):
        w0 = omega[k]
        w1 = 1.0 - w0
        if w0 > 1e-6 and w1 > 1e-6:
            m0 = mu[k] / w0
            m1 = (mu_global - mu[k]) / w1
            sigma_b_squared[k] = w0 * w1 * ((m0 - m1) ** 2)
            
    optimal_thresh = int(np.argmax(sigma_b_squared))
    
    # Binarize
    binary = np.zeros_like(image, dtype=np.uint8)
    binary[image >= optimal_thresh] = 255
    
    return optimal_thresh, binary, sigma_b_squared, hist

opt_t_scratch, otsu_bin_scratch, sigma_b, hist_vals = otsu_threshold_scratch(clean_img)

# Verify against OpenCV's in-built Otsu
ret_cv, otsu_bin_cv = cv2.threshold(clean_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print(f"Otsu Threshold (From Scratch): {opt_t_scratch}")
print(f"Otsu Threshold (OpenCV):        {int(ret_cv)}")
print(f"Scratch Otsu matches OpenCV:    {opt_t_scratch == int(ret_cv)}")


In [ ]:
# Cell 5: Visualizations
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Task 1: Denoising
axes[0, 0].imshow(noisy_sp)
axes[0, 0].set_title(f"Noisy Input (30% S&P)\nPSNR: {psnr_noisy:.2f} dB", fontweight='bold'); axes[0, 0].axis('off')

axes[0, 1].imshow(denoised_std_med)
axes[0, 1].set_title(f"Standard Median ({initial_window_size}x{initial_window_size})\nPSNR: {psnr_std:.2f} dB", fontweight='bold'); axes[0, 1].axis('off')

axes[0, 2].imshow(denoised_amf)
axes[0, 2].set_title(f"Adaptive Median ({initial_window_size} to {max_window_size})\nPSNR: {psnr_amf:.2f} dB", fontweight='bold'); axes[0, 2].axis('off')

# Task 2: Otsu Segmentation
axes[1, 0].imshow(clean_img)
axes[1, 0].set_title("Original Grayscale for Segmentation", fontweight='bold'); axes[1, 0].axis('off')

# Histogram + Between-Class Variance
ax_hist = axes[1, 1]
ax_var = ax_hist.twinx()

ax_hist.bar(range(256), hist_vals, color='steelblue', alpha=0.5, label='Histogram')
ax_var.plot(range(256), sigma_b, color='crimson', lw=2.5, label='Between-Class Variance')
ax_hist.axvline(opt_t_scratch, color='black', linestyle='--', lw=2, label=f'Otsu T={opt_t_scratch}')

ax_hist.set_title("Otsu Criterion & Histogram", fontweight='bold')
ax_hist.set_xlabel("Gray Level")
ax_hist.set_ylabel("Pixel Count", color='steelblue')
ax_var.set_ylabel(r"$\sigma_B^2$", color='crimson')

axes[1, 2].imshow(otsu_bin_scratch)
axes[1, 2].set_title(f"Otsu Segmented Binary (T={opt_t_scratch})", fontweight='bold'); axes[1, 2].axis('off')

plt.tight_layout()
plt.show()


### Conclusions
- **Adaptive Median Filter:** Outperforms standard median filtering dramatically on high-density impulse noise (delivering higher PSNR and eliminating pepper noise clumps) while preserving authentic uncorrupted pixels.
- **Otsu's Global Thresholding:** Non-parametrically discovers the mathematically optimal threshold by maximizing class separability without human calibration, aligning identically with OpenCV's internal implementation.
